# Shrub Detection — Data Preparation Pipeline

| | |
|---|---|
| **Author** | SAMI BAHIG |
| **Challenge** | Shrubwise Data Challenge |
| **Notebook** | `01_data_preparation.ipynb` |
| **Pipeline** | Setup → Config → Download → Features → Patches → Statistics → Save |

---

This notebook prepares multi-source raster patches from NAIP aerial imagery and TLS-derived shrub masks across 6 California field sites. Outputs are 64×64 px patches with 12 spectral and texture channels, ready for model training in `02_preprocessing.ipynb`.

**Data sources:**
- **NAIP** — 4-band aerial imagery (R, G, B, NIR) at 0.6 m resolution (Sprint 2)
- **3DEP** — Digital elevation model (Sprint 2)
- **TLS masks** — Ground-truth shrub labels from IntELiMon pipeline (Sprint 3)

In [ ]:
# ── CELL 1 : Environment Setup ────────────────────────────────────────────────

import subprocess
import sys

# ── Install packages ──────────────────────────────────────────────────────────
_packages = [
    # Numeric / ML
    'numpy', 'pandas', 'scipy', 'scikit-learn', 'scikit-image',
    'xgboost', 'imbalanced-learn', 'joblib', 'h5py',
    # Deep Learning
    'torch', 'albumentations',
    # Raster / GIS
    'rasterio', 'geopandas', 'shapely', 'pyproj',
    # Visualization
    'matplotlib', 'seaborn', 'ipywidgets',
    # Utilities
    'requests', 'tqdm', 'gdown',
]

print('Installing packages...')
for _pkg in _packages:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', _pkg],
        check=False
    )
print(f'  {len(_packages)} packages processed ✓')

# ── Standard library ──────────────────────────────────────────────────────────
import os
import json
import shutil
import random
import warnings
import subprocess
from pathlib import Path

# ── Numeric / ML ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter

# ── Deep Learning ─────────────────────────────────────────────────────────────
import torch

# ── Raster / GIS ──────────────────────────────────────────────────────────────
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform
from skimage.filters.rank import entropy as skentropy
from skimage.morphology import disk

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ── Utilities ─────────────────────────────────────────────────────────────────
import requests
from tqdm import tqdm

# ── Global settings ───────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Version check ─────────────────────────────────────────────────────────────
print(f'  Python   : {sys.version.split()[0]}')
print(f'  NumPy    : {np.__version__}')
print(f'  PyTorch  : {torch.__version__}')
print(f'  Rasterio : {rasterio.__version__}')
print(f'  CUDA     : {"available — " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available — using CPU"}')
print('\nEnvironment ready ✓')

Installing packages...
  22 packages processed ✓
  Python   : 3.12.0
  NumPy    : 1.26.4
  PyTorch  : 2.11.0+cu130
  Rasterio : 1.3.10
  CUDA     : not available — using CPU

Environment ready ✓


In [ ]:
# ── CELL 2 : Project Configuration ───────────────────────────────────────────
# Centralise all paths, parameters, and constants used throughout the pipeline.
# Modify only this cell to adapt the pipeline to a new dataset or experiment.

# ── Sites ─────────────────────────────────────────────────────────────────────
# Six California field sites with paired NAIP + 3DEP + TLS shrub mask data.
SITES = [
    'calaveras-big-trees',
    'dl-bliss',
    'independence-lake',
    'pacific-union-college',
    'sedgwick',
    'shaver-lake',
]

# ── Remote data root ──────────────────────────────────────────────────────────
BASE_ROOT = 'https://wifire-data.sdsc.edu/nc/public.php/dav/files'

# ── Local paths ───────────────────────────────────────────────────────────────
# All paths are relative — pipeline is fully reproducible across machines.
MASK_ROOT    = Path('./mask_outputs')   # TLS shrub masks     (Sprint 3)
# NOTE: on JupyterHub with persistent storage, replace with:
# MASK_ROOT = Path('/home/jovyan/work/_User-Persistent-Storage_CephBlock_/mask_outputs')
NAIP_ROOT    = Path('./naip')           # NAIP imagery        (Sprint 2)
DEP_ROOT     = Path('./dep')            # 3DEP elevation      (Sprint 2)
PATCH_ROOT   = Path('./patches')        # Output: image patches
FEATURE_ROOT = Path('./features')       # Output: feature arrays
FIG_ROOT     = Path('./figures')        # Output: figures & diagnostics

for _p in [NAIP_ROOT, DEP_ROOT, PATCH_ROOT, FEATURE_ROOT, FIG_ROOT]:
    _p.mkdir(parents=True, exist_ok=True)

# ── Patch extraction parameters ───────────────────────────────────────────────
# PATCH_SIZE=64 balances spatial context and GPU memory.
# PATCH_STRIDE=16 produces ~4× more patches than stride=64 (non-overlapping).
# MIN_SHRUB_RATIO=0.01 keeps near-background patches for hard-negative training.
# NOTE: increase MIN_SHRUB_RATIO (e.g. 0.05) to reduce class imbalance.
PATCH_SIZE      = 64    # pixels per patch (height = width)
PATCH_STRIDE    = 16    # sliding window stride in pixels
MIN_SHRUB_RATIO = 0.01  # minimum shrub pixel fraction to keep a patch

# ── HTTP session ──────────────────────────────────────────────────────────────
SESSION = requests.Session()

# ── Summary ───────────────────────────────────────────────────────────────────
print('Project configuration ✓')
print(f'  Sites           : {len(SITES)}')
print(f'  Patch size      : {PATCH_SIZE}×{PATCH_SIZE} px')
print(f'  Patch stride    : {PATCH_STRIDE} px')
print(f'  Min shrub ratio : {MIN_SHRUB_RATIO:.0%}')

Project configuration ✓
  Sites           : 6
  Patch size      : 64×64 px
  Patch stride    : 16 px
  Min shrub ratio : 1%


In [ ]:
# ── CELL 3 : Helper Functions ─────────────────────────────────────────────────
# Reusable utilities for file I/O, naming conventions, and site-level queries.
# All functions are stateless and depend only on constants from Cell 2.


def site_to_tif_name(site: str) -> str:
    """Convert a site slug to its corresponding NAIP/3DEP filename.

    Args:
        site: Site slug, e.g. 'calaveras-big-trees'.

    Returns:
        TIF filename, e.g. 'calaveras_big_trees.tif'.
    """
    return site.replace('-', '_') + '.tif'


def download_file(url: str, dest: Path, chunk_size: int = 1024 * 1024) -> Path:
    """Stream-download a remote file, skipping if already present on disk.

    Args:
        url        : Full URL of the remote file.
        dest       : Local destination path (parent dirs created if missing).
        chunk_size : Download chunk size in bytes (default: 1 MB).

    Returns:
        Path to the downloaded (or pre-existing) file.

    Raises:
        requests.HTTPError : If the server returns a non-2xx status code.
    """
    dest.parent.mkdir(parents=True, exist_ok=True)

    if dest.exists():
        print(f'    [skip] {dest.name} already exists')
        return dest

    with SESSION.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(dest, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True, desc=dest.name
        ) as bar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

    return dest


def list_masks_for_site(site: str) -> list[Path]:
    """Return all TLS shrub mask TIFs available for a given site.

    Args:
        site: Site slug, e.g. 'dl-bliss'.

    Returns:
        Sorted list of Path objects pointing to mask TIFs.
        Empty list if the site directory does not exist.
    """
    site_dir = MASK_ROOT / site
    if not site_dir.exists():
        return []
    return sorted(site_dir.glob('*.tif'))


# ── Smoke test ────────────────────────────────────────────────────────────────
assert site_to_tif_name('calaveras-big-trees') == 'calaveras_big_trees.tif'
assert isinstance(list_masks_for_site('nonexistent-site'), list)

print('Helper functions defined ✓')

Helper functions defined ✓


In [ ]:
# ── CELL 4 : Download NAIP Imagery ───────────────────────────────────────────
# Streams NAIP GeoTIFs from the WIFIRE remote server for all sites.
# dl-bliss requires a reprojection from EPSG:32610 → EPSG:26910 to match
# the CRS convention of all other sites. Runs once, skipped if file exists.

# ── Local overrides ───────────────────────────────────────────────────────────
# NOTE: Add entries here when a site requires a manually sourced file.
LOCAL_OVERRIDES: dict[str, Path] = {
    'dl-bliss': Path('./naip/dl_bliss_26910.tif'),
}

# ── dl-bliss reprojection (32610 → 26910) ────────────────────────────────────
# Source file was downloaded manually — see Cell 5 for full notes.
_src_bliss = Path('/home/jovyan/work/_User-Persistent-Storage_CephBlock_/naip_data/dl-bliss/dl_bliss (1).tif')
_dst_bliss = LOCAL_OVERRIDES['dl-bliss']

if _src_bliss.exists() and not _dst_bliss.exists():
    print('Reprojecting dl-bliss EPSG:32610 → EPSG:26910...')
    _dst_bliss.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(_src_bliss) as src:
        _dst_crs = rasterio.crs.CRS.from_epsg(26910)
        _transform, _width, _height = calculate_default_transform(
            src.crs, _dst_crs, src.width, src.height, *src.bounds)
        _profile = src.profile.copy()
        _profile.update(crs=_dst_crs, transform=_transform,
                        width=_width, height=_height)
        with rasterio.open(_dst_bliss, 'w', **_profile) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source        = rasterio.band(src, i),
                    destination   = rasterio.band(dst, i),
                    src_transform = src.transform,
                    src_crs       = src.crs,
                    dst_transform = _transform,
                    dst_crs       = _dst_crs,
                    resampling    = Resampling.bilinear,
                )
    print(f'  Saved → {_dst_bliss} ✓')
elif _dst_bliss.exists():
    print(f'  dl-bliss already reprojected ✓')
else:
    print(f'  WARNING: source file not found → {_src_bliss}')

# ── Download all sites ────────────────────────────────────────────────────────
print('\nDownloading NAIP imagery...')
naip_paths: dict[str, Path] = {}

for site in SITES:
    print(f'\n  [{site}]')

    if site in LOCAL_OVERRIDES:
        local = LOCAL_OVERRIDES[site]
        if local.exists():
            naip_paths[site] = local
            with rasterio.open(local) as src:
                print(f'    source  : local override')
                print(f'    bands   : {src.count} | size: {src.width}×{src.height} px | CRS: EPSG:{src.crs.to_epsg()}')
        else:
            print(f'    ERROR   : local file not found → {local.resolve()}')
        continue

    tif_name = site_to_tif_name(site)
    url      = f'{BASE_ROOT}/ucca-{site}/NAIP_3DEP_product/{tif_name}'
    dest     = NAIP_ROOT / tif_name

    try:
        download_file(url, dest)
        naip_paths[site] = dest
        with rasterio.open(dest) as src:
            print(f'    source  : remote')
            print(f'    bands   : {src.count} | size: {src.width}×{src.height} px | CRS: EPSG:{src.crs.to_epsg()}')
    except Exception as e:
        print(f'    ERROR   : {e}')

print(f'\nNAIP ready : {len(naip_paths)}/{len(SITES)} sites ✓')

  dl-bliss already reprojected ✓


  [calaveras-big-trees]
    source  : remote
    bands   : 4 | size: 7398×5375 px | CRS: EPSG:26910

  [dl-bliss]
    source  : local override
    bands   : 4 | size: 15027×18962 px | CRS: EPSG:26910

  [independence-lake]
    source  : remote
    bands   : 4 | size: 9327×7374 px | CRS: EPSG:26910

  [pacific-union-college]
    source  : remote
    bands   : 4 | size: 1540×2374 px | CRS: EPSG:26910

  [sedgwick]
    source  : remote
    bands   : 4 | size: 1743×2926 px | CRS: EPSG:26910

  [shaver-lake]
    source  : remote
    bands   : 4 | size: 1880×1773 px | CRS: EPSG:26910

NAIP ready : 6/6 sites ✓


In [ ]:
# ── CELL 5 : Notes — dl-bliss Local Override ──────────────────────────────────
# dl-bliss NAIP was not available on the WIFIRE server in the standard format.
# The file was sourced manually and stored in persistent storage.
# The LOCAL_OVERRIDES dict in Cell 4 handles this automatically.
#
# Path used:
#   /home/jovyan/work/_User-Persistent-Storage_CephBlock_/mask_outputs
#
# OPTION — update MASK_ROOT if running on JupyterHub with persistent storage:
# MASK_ROOT = Path('/home/jovyan/work/_User-Persistent-Storage_CephBlock_/mask_outputs')
#
# OPTION — dl-bliss was also attempted via Google Earth Engine export + gdown.
# See archived code below if re-downloading is needed.
#
# import ee
# aoi = ee.Geometry.Rectangle([-120.15, 38.95, -120.05, 39.05])
# naip = ee.ImageCollection('USDA/NAIP/DOQQ')\
#     .filterBounds(aoi).filterDate('2020-01-01', '2021-12-31')
# task = ee.batch.Export.image.toDrive(...)
# task.start()
#
# import gdown
# gdown.download('https://drive.google.com/uc?id=1sEUcUy13uDwdXPz8rFLc0cE98QzpmTwX',
#                './naip/dl_bliss_gee.tif', quiet=False)

print('Cell 5 : dl-bliss override documented ✓')

Cell 5 : dl-bliss override documented ✓


In [ ]:
# ── CELL 6 : Verify TLS Shrub Masks ──────────────────────────────────────────
# TLS (Terrestrial LiDAR Scanning) masks were generated by the IntELiMon
# pipeline in Sprint 3 and serve as ground-truth labels for shrub locations.
# This cell inventories available masks before feature extraction.

mask_inventory: dict[str, list[Path]] = {}

print('Verifying TLS shrub masks...')
for site in SITES:
    masks = list_masks_for_site(site)
    mask_inventory[site] = masks
    status = f'{len(masks)} mask(s)' if masks else 'WARNING — no masks found'
    print(f'  [{site}]: {status}')

total_masks = sum(len(v) for v in mask_inventory.values())
missing     = [s for s, v in mask_inventory.items() if not v]

print(f'\n  Total masks : {total_masks}')
if missing:
    print(f'  Missing     : {missing}')
else:
    print(f'  All sites covered ✓')

print('\nMask inventory complete ✓')

Verifying TLS shrub masks...
  [calaveras-big-trees]: 73 mask(s)
  [dl-bliss]: 27 mask(s)
  [independence-lake]: 56 mask(s)
  [pacific-union-college]: 37 mask(s)
  [sedgwick]: 83 mask(s)
  [shaver-lake]: 23 mask(s)

  Total masks : 299
  All sites covered ✓

Mask inventory complete ✓


## 5. Feature Engineering

From NAIP bands (R=1, G=2, B=3, NIR=4), we compute 12 spectral and texture features per pixel. Features were selected based on Sprint 1 RAP analysis and shrub-specific remote sensing literature.

| Feature        | Formula                              | Relevance                               |
|----------------|--------------------------------------|-----------------------------------------|
| NDVI           | (NIR−R) / (NIR+R)                   | Vegetation vigor — key shrub indicator  |
| EVI            | 2.5×(NIR−R) / (NIR+6R−7.5B+1)      | Enhanced vegetation, reduces soil noise |
| TGI            | G − 0.39R − 0.61B                   | Triangular Greenness (Sprint 1 RAP)     |
| NDWI           | (G−NIR) / (G+NIR)                   | Water index — excludes water bodies     |
| Brightness     | mean(R, G, B, NIR)                  | Overall reflectance                     |
| VARI           | (G−R) / (G+R−B)                     | Visible Atm. Resistance (IntELiMon)     |
| Texture (var)  | local NDVI variance 5×5             | Surface roughness — shrubs are textured |
| Texture (ent)  | local NIR entropy disk(3)           | Complexity — shrubs vs bare ground      |

In [ ]:
# ── CELL 8 : compute_features ─────────────────────────────────────────────────
# Computes 12 spectral and texture features per pixel from NAIP bands.
# All indices use epsilon stabilisation to avoid division-by-zero.
# VARI is included for IntELiMon compatibility (not in Markdown table above).

def compute_features(naip_path: Path) -> tuple[dict, dict, object, object]:
    """Compute spectral indices and texture features from a NAIP GeoTIFF.

    Reads all 4 NAIP bands (R, G, B, NIR) and derives 12 features:
    4 raw bands + 6 spectral indices + 2 texture descriptors.
    If NIR band is absent (< 4 bands), it is substituted with zeros.

    Args:
        naip_path: Path to a 4-band NAIP GeoTIFF (band order: R, G, B, NIR).

    Returns:
        features  : Dict mapping feature name → 2D float32 array (H, W).
        profile   : Rasterio dataset profile (CRS, transform, dtype, etc.).
        transform : Affine transform of the source raster.
        crs       : Coordinate reference system of the source raster.
    """
    with rasterio.open(naip_path) as src:
        R         = src.read(1).astype(np.float32)
        G         = src.read(2).astype(np.float32)
        B         = src.read(3).astype(np.float32)
        NIR       = src.read(4).astype(np.float32) if src.count >= 4 \
                    else np.zeros_like(R)
        profile   = src.profile
        transform = src.transform
        crs       = src.crs

    eps = 1e-8  # numerical stability for all ratio-based indices

    # ── Spectral indices ──────────────────────────────────────────────────────
    ndvi       = (NIR - R)   / (NIR + R + eps)
    evi        = 2.5 * (NIR - R) / (NIR + 6*R - 7.5*B + 1 + eps)
    tgi        = G - 0.39*R - 0.61*B                    # Sprint 1 RAP metric
    ndwi       = (G - NIR)   / (G + NIR + eps)
    brightness = (R + G + B + NIR) / 4.0
    vari       = (G - R)     / (G + R - B + eps)        # IntELiMon compatibility

    # ── Texture features ──────────────────────────────────────────────────────
    # Variance: local NDVI variance over 5×5 window (shrubs → high variance)
    ndvi_u      = uniform_filter(ndvi,    size=5)
    ndvi_u2     = uniform_filter(ndvi**2, size=5)
    texture_var = (ndvi_u2 - ndvi_u**2).astype(np.float32)

    # Entropy: local NIR entropy over disk(3) (shrubs → high complexity)
    nir_uint8   = ((NIR - NIR.min()) / (NIR.max() - NIR.min() + eps) * 255
                   ).astype(np.uint8)
    texture_ent = skentropy(nir_uint8, disk(3)).astype(np.float32)

    # ── Feature dict ──────────────────────────────────────────────────────────
    features = {
        'R': R, 'G': G, 'B': B, 'NIR': NIR,
        'ndvi'       : ndvi,
        'evi'        : evi,
        'tgi'        : tgi,
        'ndwi'       : ndwi,
        'brightness' : brightness,
        'vari'       : vari,
        'texture_var': texture_var,
        'texture_ent': texture_ent,
    }
    return features, profile, transform, crs


print(f'compute_features defined — 12 features: '
      f'4 bands + 6 spectral indices + 2 texture descriptors ✓')

compute_features defined — 12 features: 4 bands + 6 spectral indices + 2 texture descriptors ✓


In [ ]:
# ── CELL 9 : Smoke Test — compute_features ────────────────────────────────────
# Minimal validation run on one site to catch errors early, before processing
# all sites. Checks output shape, feature names, CRS, and NDVI range [-1, 1].

test_site = [s for s in SITES if naip_paths.get(s)][0]
print(f'Test site : {test_site}')

test_features, test_profile, test_transform, test_crs = \
    compute_features(naip_paths[test_site])

# ── Output validation ─────────────────────────────────────────────────────────
H, W = test_features['R'].shape
print(f'  Features : {list(test_features.keys())}')
print(f'  Shape    : {H}×{W} px')
print(f'  CRS      : EPSG:{test_crs.to_epsg()}')

# ── NDVI sanity check (expected range: -1 to 1) ───────────────────────────────
ndvi = test_features['ndvi']
print(f'\n  NDVI — min: {ndvi.min():.3f} | max: {ndvi.max():.3f} | mean: {ndvi.mean():.3f}')
assert -1.0 <= ndvi.min() and ndvi.max() <= 1.0, \
    f'NDVI out of expected range [-1, 1]: [{ndvi.min():.3f}, {ndvi.max():.3f}]'

print('\nSmoke test passed ✓')

Test site : calaveras-big-trees
  Features : ['R', 'G', 'B', 'NIR', 'ndvi', 'evi', 'tgi', 'ndwi', 'brightness', 'vari', 'texture_var', 'texture_ent']
  Shape    : 5375×7398 px
  CRS      : EPSG:26910

  NDVI — min: -0.913 | max: 0.832 | mean: 0.133

Smoke test passed ✓


## 6. Exploratory Visualization

Visual inspection of NAIP imagery, derived feature maps, and TLS shrub masks for a representative site. Confirms spatial alignment between inputs and labels before patch extraction.

In [ ]:
# ── CELL 11 : visualize_site_features ────────────────────────────────────────
# Plots NAIP RGB composite + 7 feature maps in a 2×4 grid.
# Saves figure to FIG_ROOT for reporting and visual QA.

def visualize_site_features(site: str, features: dict, save: bool = True) -> None:
    """Plot NAIP RGB composite and all derived feature maps for a site.

    Produces a 2×4 grid: RGB + NDVI, EVI, TGI, NDWI, Brightness,
    Texture (variance), Texture (entropy).

    Args:
        site    : Site slug, used for figure title and filename.
        features: Feature dict returned by compute_features().
        save    : If True, saves figure to FIG_ROOT/{site}_features.png.
    """
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(f'Feature Maps — {site}', fontsize=16, fontweight='bold')

    # ── RGB composite ─────────────────────────────────────────────────────────
    def _norm(x: np.ndarray) -> np.ndarray:
        """Min-max normalise to [0, 1] for display."""
        return (x - x.min()) / (x.max() - x.min() + 1e-8)

    rgb = np.stack([_norm(features['R']),
                    _norm(features['G']),
                    _norm(features['B'])], axis=-1)
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title('NAIP RGB', fontweight='bold')
    axes[0, 0].axis('off')

    # ── Feature maps ──────────────────────────────────────────────────────────
    _features = ['ndvi', 'evi', 'tgi', 'ndwi', 'brightness', 'texture_var', 'texture_ent']
    _cmaps    = ['RdYlGn', 'RdYlGn', 'Greens', 'Blues', 'gray', 'hot', 'viridis']
    _titles   = ['NDVI', 'EVI', 'TGI', 'NDWI', 'Brightness', 'Texture (var)', 'Texture (ent)']

    for idx, (fname, cmap, title) in enumerate(zip(_features, _cmaps, _titles)):
        ax = axes.flatten()[idx + 1]
        im = ax.imshow(features[fname], cmap=cmap)
        ax.set_title(title, fontweight='bold')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()

    if save:
        path = FIG_ROOT / f'{site}_features.png'
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f'  Saved : {path} ✓')

    plt.show()


# ── Run on test site ──────────────────────────────────────────────────────────
visualize_site_features(test_site, test_features)

Saved : figures/calaveras-big-trees_features.png ✓


## 7. Mask Alignment & Patch Extraction

For each (NAIP tile, TLS mask) pair:
1. **Reproject** the TLS mask to match the NAIP CRS, resolution, and extent
2. **Extract** sliding-window patches of size 64×64 px (stride=16)
3. **Filter** patches with shrub coverage < MIN_SHRUB_RATIO (1%) to reduce class imbalance while retaining hard negatives for robust training

In [ ]:
# ── CELL 13 : Mask Alignment & Patch Extraction ───────────────────────────────
# Two core functions for the data preparation pipeline:
#   1. align_mask_to_naip  — reprojects TLS mask to NAIP pixel space
#   2. extract_patches     — sliding-window patch extraction with shrub filter


def align_mask_to_naip(mask_path: Path, naip_path: Path) -> np.ndarray:
    """Reproject a TLS shrub mask GeoTIFF to match the NAIP raster grid exactly.

    Reprojects the mask to the NAIP CRS, resolution, and spatial extent using
    nearest-neighbour resampling to preserve binary label integrity.

    Args:
        mask_path : Path to the TLS shrub mask GeoTIFF (any CRS).
        naip_path : Path to the reference NAIP GeoTIFF.

    Returns:
        Binary uint8 array of shape (H, W) in NAIP pixel space.
        Pixels with value 1 indicate shrub presence.
    """
    with rasterio.open(naip_path) as naip_src:
        naip_crs       = naip_src.crs
        naip_transform = naip_src.transform
        naip_width     = naip_src.width
        naip_height    = naip_src.height

    with rasterio.open(mask_path) as mask_src:
        mask_data      = mask_src.read(1).astype(np.float32)
        mask_crs       = mask_src.crs
        mask_transform = mask_src.transform

    # Nearest-neighbour resampling preserves binary values (0/1)
    aligned = np.zeros((naip_height, naip_width), dtype=np.float32)
    reproject(
        source        = mask_data,
        destination   = aligned,
        src_transform = mask_transform,
        src_crs       = mask_crs,
        dst_transform = naip_transform,
        dst_crs       = naip_crs,
        resampling    = Resampling.nearest,
    )
    return (aligned > 0).astype(np.uint8)


def extract_patches(
    feature_stack   : np.ndarray,
    label_map       : np.ndarray,
    patch_size      : int   = 64,
    stride          : int   = 32,
    min_shrub_ratio : float = 0.05,
) -> tuple[np.ndarray, np.ndarray]:
    """Extract paired (features, label) patches via sliding window.

    Iterates over the full image with a sliding window and retains only patches
    whose shrub pixel fraction meets the minimum threshold.

    Args:
        feature_stack   : Input features, shape (H, W, C).
        label_map       : Binary shrub mask, shape (H, W).
        patch_size      : Spatial size of each square patch in pixels.
        stride          : Step between consecutive patch origins in pixels.
        min_shrub_ratio : Minimum fraction of shrub pixels to retain a patch.
                          NOTE: set to 0.0 to keep all patches (debug mode).

    Returns:
        patches_x : Float32 array of shape (N, patch_size, patch_size, C).
        patches_y : Uint8 array of shape   (N, patch_size, patch_size).
        Returns two empty arrays if no patches pass the shrub filter.
    """
    H, W, C   = feature_stack.shape
    patches_x = []
    patches_y = []

    for r in range(0, H - patch_size + 1, stride):
        for c in range(0, W - patch_size + 1, stride):
            py = label_map[r:r+patch_size, c:c+patch_size]
            if py.mean() >= min_shrub_ratio:
                patches_x.append(feature_stack[r:r+patch_size, c:c+patch_size, :])
                patches_y.append(py)

    if not patches_x:
        return np.array([]), np.array([])

    return np.stack(patches_x), np.stack(patches_y)


print('align_mask_to_naip  defined ✓')
print('extract_patches     defined ✓')

align_mask_to_naip  defined ✓
extract_patches     defined ✓


In [ ]:
# ── CELL 14 : Feature Stack Builder ──────────────────────────────────────────
# Assembles all feature arrays into a single (H, W, C) tensor.
# Channel order is fixed here and must remain consistent across all pipeline
# stages (training, inference, evaluation).

# ── Canonical channel order ───────────────────────────────────────────────────
# Defined once as a constant — single source of truth for all downstream code.
CHANNEL_NAMES: list[str] = [
    'R', 'G', 'B', 'NIR',          # raw NAIP bands
    'ndvi', 'evi', 'tgi', 'ndwi',  # spectral indices
    'brightness', 'vari',           # broadband & ratio
    'texture_var', 'texture_ent',   # texture descriptors
]
IN_CHANNELS: int = len(CHANNEL_NAMES)


def build_feature_stack(features: dict) -> tuple[np.ndarray, list[str]]:
    """Stack all feature arrays into a single (H, W, C) float32 array.

    Channel order follows CHANNEL_NAMES exactly, ensuring reproducibility
    across training, inference, and evaluation.

    Args:
        features: Feature dict returned by compute_features().

    Returns:
        stack         : Float32 array of shape (H, W, C).
        channel_names : Ordered list of channel names (== CHANNEL_NAMES).

    Raises:
        KeyError: If a required channel is missing from the features dict.
    """
    missing = [k for k in CHANNEL_NAMES if k not in features]
    if missing:
        raise KeyError(f'Missing features: {missing}')

    stack = np.stack([features[k] for k in CHANNEL_NAMES], axis=-1)
    return stack.astype(np.float32), CHANNEL_NAMES


print(f'build_feature_stack defined ✓')
print(f'  Channels ({IN_CHANNELS}): {CHANNEL_NAMES}')

build_feature_stack defined ✓
  Channels (12): ['R', 'G', 'B', 'NIR', 'ndvi', 'evi', 'tgi', 'ndwi', 'brightness', 'vari', 'texture_var', 'texture_ent']


In [ ]:
# ── CELL 15 : Main Pipeline — Feature Extraction & Patch Building ─────────────
# Processes all sites end-to-end:
#   1. Compute 12-channel feature stack from NAIP
#   2. Align all TLS masks to NAIP grid and merge via union
#   3. Extract sliding-window patches and filter by shrub coverage
#   4. Accumulate patches across all sites into X_all, Y_all

all_patches_x : list[np.ndarray] = []
all_patches_y : list[np.ndarray] = []
site_stats    : list[dict]       = []

for site in SITES:
    print(f'\n  {"═"*60}')
    print(f'  {site}')
    print(f'  {"═"*60}')

    if site not in naip_paths:
        print(f'  [skip] no NAIP file available')
        continue

    masks = mask_inventory.get(site, [])
    if not masks:
        print(f'  [skip] no TLS masks available')
        continue

    try:
        print(f'  Computing features...')
        features, profile, transform, crs = compute_features(naip_paths[site])
        feature_stack, _ = build_feature_stack(features)
        H, W, C = feature_stack.shape
        print(f'  Feature stack : {H}×{W}×{C}')

        label_map  = np.zeros((H, W), dtype=np.uint8)
        n_masks_ok = 0

        for mask_path in tqdm(masks, desc='  Aligning masks', leave=False):
            try:
                aligned   = align_mask_to_naip(mask_path, naip_paths[site])
                label_map = np.maximum(label_map, aligned)
                n_masks_ok += 1
            except Exception as e:
                print(f'    WARN : {mask_path.name} — {e}')

        shrub_pct = label_map.mean() * 100
        print(f'  Masks aligned : {n_masks_ok}/{len(masks)}')
        print(f'  Shrub cover   : {shrub_pct:.2f}% of NAIP tile')

        px, py = extract_patches(
            feature_stack, label_map,
            patch_size      = PATCH_SIZE,
            stride          = PATCH_STRIDE,
            min_shrub_ratio = MIN_SHRUB_RATIO,
        )

        if px.size == 0:
            print(f'  [skip] no patches passed shrub filter')
            continue

        print(f'  Patches kept  : {len(px)}')
        all_patches_x.append(px)
        all_patches_y.append(py)
        np.save(FEATURE_ROOT / f'{site}_label_map.npy', label_map)

        site_stats.append({
            'site'      : site,
            'n_masks'   : n_masks_ok,
            'shrub_pct' : round(shrub_pct, 4),
            'n_patches' : len(px),
            'image_size': f'{H}×{W}',
        })

    except Exception as e:
        print(f'  [ERROR] {e} — skipping site')
        continue

# ── Concatenate all patches ───────────────────────────────────────────────────
if all_patches_x:
    X_all = np.concatenate(all_patches_x, axis=0)
    Y_all = np.concatenate(all_patches_y, axis=0)

    print(f'\n  {"═"*60}')
    print(f'  PIPELINE COMPLETE')
    print(f'  {"═"*60}')
    print(f'  Sites processed : {len(site_stats)}/{len(SITES)}')
    print(f'  Total patches   : {len(X_all):,}')
    print(f'  X shape         : {X_all.shape}  (N, H, W, C)')
    print(f'  Y shape         : {Y_all.shape}  (N, H, W)')
    print(f'\n  Per-site breakdown:')
    for s in site_stats:
        print(f'    {s["site"]:<30} {s["n_patches"]:>5} patches | '
              f'{s["shrub_pct"]:>5.2f}% shrub | {s["image_size"]}')
else:
    print('WARNING: no patches extracted from any site — check NAIP and mask paths.')


  ════════════════════════════════════════════════════════════
  calaveras-big-trees
  ════════════════════════════════════════════════════════════
  Computing features...
  Feature stack : 5375×7398×12
  Masks aligned : 73/73
  Shrub cover   : 0.06% of NAIP tile
  Patches kept  : 1768

  ════════════════════════════════════════════════════════════
  dl-bliss
  ════════════════════════════════════════════════════════════
  Computing features...
  Feature stack : 18962×15027×12
  Masks aligned : 27/27
  Shrub cover   : 0.00% of NAIP tile
  Patches kept  : 755

  ════════════════════════════════════════════════════════════
  independence-lake
  ════════════════════════════════════════════════════════════
  Computing features...
  Feature stack : 7374×9327×12
  Masks aligned : 56/56
  Shrub cover   : 0.02% of NAIP tile
  Patches kept  : 1025

  ════════════════════════════════════════════════════════════
  pacific-union-college
  ══════════════════════════════════════════════════════════

In [ ]:
# ── CELL 16 : Dataset Validation ─────────────────────────────────────────────
# Final sanity check on the assembled dataset before saving.
# Verifies shapes, dtypes, label distribution, and per-site coverage.

assert len(X_all) > 0, 'X_all is empty — check pipeline output (Cell 15)'
assert X_all.shape[0] == Y_all.shape[0], 'Mismatch between X and Y patch counts'
assert X_all.shape[-1] == IN_CHANNELS, \
    f'Expected {IN_CHANNELS} channels, got {X_all.shape[-1]}'

shrub_pixels = int(Y_all.sum())
total_pixels = Y_all.size
shrub_ratio  = shrub_pixels / total_pixels

print('Dataset validation ✓')
print(f'  Sites processed : {len(site_stats)}/{len(SITES)}')
print(f'  Total patches   : {len(X_all):,}')
print(f'  X shape         : {X_all.shape}  (N, H, W, C)')
print(f'  Y shape         : {Y_all.shape}  (N, H, W)')
print(f'  X dtype         : {X_all.dtype}')
print(f'  Y dtype         : {Y_all.dtype}')
print(f'  Shrub pixels    : {shrub_pixels:,} / {total_pixels:,} ({shrub_ratio:.2%})')
print(f'  Class balance   : {shrub_ratio:.2%} shrub / {1-shrub_ratio:.2%} background')

Dataset validation ✓
  Sites processed : 6/6
  Total patches   : 6,566
  X shape         : (6566, 64, 64, 12)  (N, H, W, C)
  Y shape         : (6566, 64, 64)  (N, H, W)
  X dtype         : float32
  Y dtype         : uint8
  Shrub pixels    : 1,297,390 / 26,894,336 (4.82%)
  Class balance   : 4.82% shrub / 95.18% background


In [ ]:
# ── CELL 17 : Deduplicate & Save Dataset ──────────────────────────────────────
# Removes duplicate site entries (can occur if pipeline was re-run partially),
# then saves the clean dataset to both local and persistent storage.
# NOTE: PATCH_ROOT_PERSIST is the JupyterHub persistent volume path.
#       For portability (Colab, Docker), use PATCH_ROOT instead.

# ── Deduplicate site_stats ────────────────────────────────────────────────────
seen             : set  = set()
site_stats_clean : list = []

for s in site_stats:
    if s['site'] not in seen:
        seen.add(s['site'])
        site_stats_clean.append(s)

total_clean  = sum(s['n_patches'] for s in site_stats_clean)
X_all_clean  = X_all[:total_clean]
Y_all_clean  = Y_all[:total_clean]

print(f'Deduplication ✓')
print(f'  Sites   : {len(site_stats_clean)}')
print(f'  Patches : {total_clean:,}')

# ── Save to persistent storage ────────────────────────────────────────────────
# NOTE: replace with PATCH_ROOT for local/Colab/Docker runs
PATCH_ROOT_PERSIST = Path(
    '/home/jovyan/work/_User-Persistent-Storage_CephBlock_/sprint4/patches'
)
PATCH_ROOT_PERSIST.mkdir(parents=True, exist_ok=True)

np.save(PATCH_ROOT_PERSIST / 'X_patches.npy', X_all_clean)
np.save(PATCH_ROOT_PERSIST / 'Y_patches.npy', Y_all_clean)

with open(PATCH_ROOT_PERSIST / 'channel_names.json', 'w') as f:
    json.dump(CHANNEL_NAMES, f)

pd.DataFrame(site_stats_clean).to_csv(
    PATCH_ROOT_PERSIST / 'site_stats.csv', index=False
)

# ── Also save to local PATCH_ROOT for portability ─────────────────────────────
np.save(PATCH_ROOT / 'X_patches.npy', X_all_clean)
np.save(PATCH_ROOT / 'Y_patches.npy', Y_all_clean)

print(f'\nSaved to persistent storage ✓')
print(f'  Path : {PATCH_ROOT_PERSIST}')
print(f'  X    : {X_all_clean.shape}  ({X_all_clean.nbytes / 1e6:.1f} MB)')
print(f'  Y    : {Y_all_clean.shape}  ({Y_all_clean.nbytes / 1e6:.1f} MB)')

Deduplication ✓
  Sites   : 6
  Patches : 6,566

Saved to persistent storage ✓
  Path : /home/jovyan/work/_User-Persistent-Storage_CephBlock_/sprint4/patches
  X    : (6566, 64, 64, 12)  (202.5 MB)
  Y    : (6566, 64, 64)  (26.9 MB)


## 8. Dataset Statistics & Class Balance Analysis

Quantifies class imbalance, per-site patch distribution, and feature statistics across the full dataset. These metrics inform training decisions such as loss weighting, oversampling strategy, and data augmentation intensity.

In [ ]:
# ── CELL 19 : Dataset Statistics & Class Balance Analysis ─────────────────────
# Computes per-site patch counts, shrub coverage, and global class imbalance.
# Results inform loss weighting and oversampling strategy in model training.

# ── Per-site summary table ────────────────────────────────────────────────────
df_stats = pd.DataFrame(site_stats_clean)
print('Per-site statistics:')
print(df_stats.to_string(index=False))
df_stats.to_csv(FEATURE_ROOT / 'site_stats.csv', index=False)

# ── Global class balance ──────────────────────────────────────────────────────
total_pixels    = Y_all_clean.size
shrub_pixels    = int(Y_all_clean.sum())
bg_pixels       = total_pixels - shrub_pixels
shrub_pct       = 100 * shrub_pixels / total_pixels
imbalance_ratio = bg_pixels / shrub_pixels if shrub_pixels > 0 else float('inf')

print(f'\nClass balance:')
print(f'  Total pixels    : {total_pixels:,}')
print(f'  Shrub pixels    : {shrub_pixels:,}  ({shrub_pct:.1f}%)')
print(f'  Background px   : {bg_pixels:,}  ({100 - shrub_pct:.1f}%)')
print(f'  Imbalance ratio : 1:{imbalance_ratio:.0f}  (background:shrub)')

# ── Training recommendations ──────────────────────────────────────────────────
print(f'\nTraining recommendations:')
if imbalance_ratio > 10:
    print(f'  ⚠ Severe imbalance (1:{imbalance_ratio:.0f})')
    print(f'    → Apply pos_weight={imbalance_ratio:.1f} in BCEWithLogitsLoss')
    print(f'    → Consider SMOTE for classical ML baselines')
    print(f'    → Use Dice or Focal loss for deep learning models')
elif imbalance_ratio > 5:
    print(f'  ⚠ Moderate imbalance (1:{imbalance_ratio:.0f})')
    print(f'    → Apply pos_weight in BCEWithLogitsLoss')
else:
    print(f'  ✓ Class balance acceptable (1:{imbalance_ratio:.0f})')

Per-site statistics:
                  site  n_masks  shrub_pct  n_patches    image_size
   calaveras-big-trees       73     0.0593       1768   5375×7398
              dl-bliss       27     0.0026        755   18962×15027
     independence-lake       56     0.0160       1025   7374×9327
 pacific-union-college       37     0.2392        814   2374×1540
              sedgwick       83     0.5025       1503   2926×1743
           shaver-lake       23     0.2427        701   1773×1880

Class balance:
  Total pixels    : 26,894,336
  Shrub pixels    : 1,297,390  (4.8%)
  Background px   : 25,596,946  (95.2%)
  Imbalance ratio : 1:20  (background:shrub)

Training recommendations:
  ⚠ Severe imbalance (1:20)
    → Apply pos_weight=20.0 in BCEWithLogitsLoss
    → Consider SMOTE for classical ML baselines
    → Use Dice or Focal loss for deep learning models


In [ ]:
# ── CELL 20 : Class Balance Visualization ─────────────────────────────────────
# Two-panel figure: class distribution pie chart + patches-per-site bar chart.
# Saved to FIG_ROOT for reporting.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dataset Overview', fontsize=14, fontweight='bold')

# ── Panel 1 : Class distribution ─────────────────────────────────────────────
axes[0].pie(
    [shrub_pixels, bg_pixels],
    labels     = ['Shrub', 'Background'],
    colors     = ['#e74c3c', '#2ecc71'],
    autopct    = '%1.1f%%',
    startangle = 90,
)
axes[0].set_title('Class Distribution (all patches)', fontweight='bold')

# ── Panel 2 : Patches per site ────────────────────────────────────────────────
df_plot = pd.DataFrame(site_stats_clean).sort_values('n_patches', ascending=True)
axes[1].barh(df_plot['site'], df_plot['n_patches'], color='steelblue', edgecolor='white')
axes[1].set_xlabel('Number of Patches')
axes[1].set_title('Training Patches per Site', fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

for i, (val, site) in enumerate(zip(df_plot['n_patches'], df_plot['site'])):
    axes[1].text(val + 5, i, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIG_ROOT / 'class_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved : {FIG_ROOT}/class_balance.png ✓')

Saved : figures/class_balance.png ✓


## 9. Sample Patch Visualization

Visual inspection of 8 randomly sampled training patches. Each column shows the same patch across three representations: RGB composite, NDVI map, and ground-truth shrub mask. Confirms spatial alignment between features and labels.

In [ ]:
# ── CELL 21 : Sample Patch Visualization ─────────────────────────────────────
# Displays 8 random patches × 3 rows (RGB / NDVI / shrub mask).
# Confirms visual alignment between input features and ground-truth labels.

def _norm(x: np.ndarray) -> np.ndarray:
    """Min-max normalise to [0, 1] for display."""
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

n_show = min(8, len(X_all_clean))
idxs   = np.random.choice(len(X_all_clean), n_show, replace=False)

fig, axes = plt.subplots(3, n_show, figsize=(n_show * 2.5, 7))
fig.suptitle('Sample Training Patches', fontsize=14, fontweight='bold')

for col, idx in enumerate(idxs):
    patch = X_all_clean[idx]
    label = Y_all_clean[idx]

    # ── Row 0 : RGB ───────────────────────────────────────────────────────────
    rgb = np.stack([_norm(patch[:, :, 0]),
                    _norm(patch[:, :, 1]),
                    _norm(patch[:, :, 2])], axis=-1)
    axes[0, col].imshow(rgb)
    axes[0, col].set_title(f'#{idx}', fontsize=8)
    axes[0, col].axis('off')

    # ── Row 1 : NDVI (channel 4) ──────────────────────────────────────────────
    axes[1, col].imshow(patch[:, :, 4], cmap='RdYlGn', vmin=-1, vmax=1)
    axes[1, col].axis('off')

    # ── Row 2 : Shrub mask ────────────────────────────────────────────────────
    axes[2, col].imshow(label, cmap='Reds', vmin=0, vmax=1)
    axes[2, col].axis('off')

for row, row_label in enumerate(['RGB', 'NDVI', 'Shrub Mask']):
    axes[row, 0].set_ylabel(row_label, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_ROOT / 'sample_patches.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved : {FIG_ROOT}/sample_patches.png ✓')

Saved : figures/sample_patches.png ✓


## 10. Save Preprocessed Data

Save all processed data and reproducibility artifacts for use in `02_preprocessing.ipynb` and `03_training.ipynb`.

In [ ]:
# ── CELL 23 : Save Preprocessed Data & Reproducibility Artifacts ─────────────
# Final outputs for downstream notebooks (02_preprocessing, 03_training):
#   - X_patches.npy     : feature patches  (N, H, W, C)
#   - Y_patches.npy     : label patches    (N, H, W)
#   - channel_names.json: channel order    (C,)
#   - site_stats.csv    : per-site metrics
#   - requirements.txt  : pip environment snapshot

# ── Disk usage check ──────────────────────────────────────────────────────────
total, used, free = shutil.disk_usage('/')
print('Disk usage:')
print(f'  Total : {total / 1024**3:.1f} GB')
print(f'  Used  : {used  / 1024**3:.1f} GB')
print(f'  Free  : {free  / 1024**3:.1f} GB')

# ── requirements.txt snapshot ─────────────────────────────────────────────────
result = subprocess.run(['pip', 'freeze'], capture_output=True, text=True)
req_path = Path('./requirements.txt')
req_path.write_text(result.stdout)
print(f'\nrequirements.txt saved ✓')

# ── Final dataset summary ─────────────────────────────────────────────────────
print(f'\nDataset ready for training:')
print(f'  X_patches : {X_all_clean.shape}  ({X_all_clean.nbytes / 1e6:.1f} MB)')
print(f'  Y_patches : {Y_all_clean.shape}  ({Y_all_clean.nbytes / 1e6:.1f} MB)')
print(f'  Channels  : {CHANNEL_NAMES}')
print(f'\nData preparation pipeline complete ✓')

Disk usage:
  Total : 500.0 GB
  Used  : 312.4 GB
  Free  : 187.6 GB

requirements.txt saved ✓

Dataset ready for training:
  X_patches : (6566, 64, 64, 12)  (202.5 MB)
  Y_patches : (6566, 64, 64)  (26.9 MB)
  Channels  : ['R', 'G', 'B', 'NIR', 'ndvi', 'evi', 'tgi', 'ndwi', 'brightness', 'vari', 'texture_var', 'texture_ent']

Data preparation pipeline complete ✓
